# **Create a Cryptocurrency Trading Algorithm in Python**

> **IBM Skills Network — Guided Project**  
> Original author: [David Pasternak](https://www.linkedin.com/in/david-pasternak-6b84a2208/)  
> Completed and extended for portfolio by: Rounak

---

## Overview

Cryptocurrencies differ from fiat currencies (USD, EUR) in that they have no central regulatory authority — transactions are secured cryptographically in a distributed ledger. This project builds a **Moving Average Crossover** algorithmic trading strategy on Bitcoin–USD historical data, backtests it, and compares its performance to a simple Buy-and-Hold baseline.

**What you will learn:**
- Fetch cryptocurrency market data with `yfinance`
- Perform basic market analysis using Simple Moving Averages (SMA)
- Implement a Moving Average Crossover trading algorithm
- Backtest and visualize strategy performance


---
## 1. Setup & Imports

In [ ]:
# Install yfinance if not already installed
!pip install yfinance --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.dates import DateFormatter

print("All libraries imported successfully.")

---
## 2. Fetch Bitcoin–USD Historical Data

In [ ]:
# Download BTC-USD daily data for the year 2020
BTC_USD = yf.download("BTC-USD", start='2020-01-01', end='2020-12-31', interval='1d')
print(f"Downloaded {len(BTC_USD)} rows of BTC-USD data.")

In [ ]:
# Preview the first 5 rows
BTC_USD.head()

The dataframe columns are:
- **Open** — price at market open
- **High / Low** — day's highest and lowest price
- **Close / Adj Close** — closing and adjusted closing price
- **Volume** — total volume traded

---
## 3. Price Chart

In [ ]:
fig, ax = plt.subplots(dpi=150)

date_format = DateFormatter("%b-%d-%y")
ax.xaxis.set_major_formatter(date_format)
ax.tick_params(axis='x', labelsize=8)
fig.autofmt_xdate()

ax.plot(BTC_USD['Close'], lw=0.75, color='steelblue', label='Closing Price')

ax.set_ylabel('Price of Bitcoin (USD)')
ax.set_title('Bitcoin to USD Exchange Rate — 2020')
ax.grid(alpha=0.4)
ax.legend()

plt.tight_layout()
plt.savefig('charts/01_btc_price_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Simple Moving Averages (SMA)

A **Simple Moving Average** smooths out short-term price noise by averaging the closing price over a rolling window.

| Day   | 1 | 2 | 3 | 4 | 5  | 6  | 7 | 8 | 9 |
|-------|---|---|---|---|----|----|---|---|---|
| Value | 4 | 3 | 5 | 7 | 10 | 11 | 9 | 7 | 6 |
| 3-day SMA | n/a | n/a | 4 | 5 | 7.33 | 9.33 | 10 | 9 | 7.33 |

In [ ]:
# 9-day SMA
BTC_USD['SMA_9'] = BTC_USD['Close'].rolling(window=9, min_periods=1).mean()

# 30-day SMA
BTC_USD['SMA_30'] = BTC_USD['Close'].rolling(window=30, min_periods=1).mean()

BTC_USD.tail()

In [ ]:
fig, ax = plt.subplots(dpi=150)

date_format = DateFormatter("%b-%d-%y")
ax.xaxis.set_major_formatter(date_format)
ax.tick_params(axis='x', labelsize=8)
fig.autofmt_xdate()

ax.plot(BTC_USD['Close'],  lw=0.75, label='Closing Price')
ax.plot(BTC_USD['SMA_9'],  lw=0.75, alpha=0.8, label='9-Day SMA')
ax.plot(BTC_USD['SMA_30'], lw=0.75, alpha=0.8, label='30-Day SMA')

ax.set_ylabel('Price of Bitcoin (USD)')
ax.set_title('BTC-USD with Simple Moving Averages — 2020')
ax.grid(alpha=0.4)
ax.legend()

plt.tight_layout()
plt.savefig('charts/02_btc_sma_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Moving Average Crossover Strategy

**Logic:**
- **Buy** when the short-term SMA crosses *above* the long-term SMA (bullish signal)
- **Sell** when the short-term SMA crosses *below* the long-term SMA (bearish signal)
- Hold cash when not in a position

In [ ]:
# Create trade signals dataframe
trade_signals = pd.DataFrame(index=BTC_USD.index)

short_interval = 10
long_interval  = 40

trade_signals['Short'] = BTC_USD['Close'].rolling(window=short_interval, min_periods=1).mean()
trade_signals['Long']  = BTC_USD['Close'].rolling(window=long_interval,  min_periods=1).mean()

# Signal: 1 when short SMA > long SMA, else 0
trade_signals['Signal'] = 0.0
trade_signals['Signal'] = np.where(trade_signals['Short'] > trade_signals['Long'], 1.0, 0.0)

# Position: diff of signal column gives +1 (buy) and -1 (sell) at crossover points
trade_signals['Position'] = trade_signals['Signal'].diff()

trade_signals.head(10)

---
## 6. Visualise Buy / Sell Signals

In [ ]:
fig, ax = plt.subplots(dpi=150)

date_format = DateFormatter("%b-%d-%y")
ax.xaxis.set_major_formatter(date_format)
ax.tick_params(axis='x', labelsize=8)
fig.autofmt_xdate()

ax.plot(BTC_USD['Close'],       lw=0.75,              label='Closing Price')
ax.plot(trade_signals['Short'], lw=0.75, alpha=0.75, color='orange', label=f'{short_interval}-Day SMA (Short)')
ax.plot(trade_signals['Long'],  lw=0.75, alpha=0.75, color='purple', label=f'{long_interval}-Day SMA (Long)')

# Buy signals — green upward carets
buy_signals  = trade_signals[trade_signals['Position'] ==  1.0]
sell_signals = trade_signals[trade_signals['Position'] == -1.0]

ax.plot(buy_signals.index,  trade_signals.Short[trade_signals['Position'] ==  1.0],
        marker=6, ms=5, linestyle='none', color='green', label='Buy Signal')
ax.plot(sell_signals.index, trade_signals.Short[trade_signals['Position'] == -1.0],
        marker=7, ms=5, linestyle='none', color='red',   label='Sell Signal')

ax.set_ylabel('Price of Bitcoin (USD)')
ax.set_title('Moving Average Crossover — Buy/Sell Signals 2020')
ax.grid(alpha=0.4)
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('charts/03_btc_signals_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Backtest the Strategy

In [ ]:
initial_balance = 10_000.0  # Starting capital in USD

backtest = pd.DataFrame(index=trade_signals.index)

# Daily % return of BTC
backtest['BTC_Return'] = BTC_USD['Close'] / BTC_USD['Close'].shift(1)

# Algorithm return: use BTC return when holding BTC (Signal==1), else 1.0 (holding cash)
backtest['Alg_Return'] = np.where(trade_signals.Signal == 1, backtest.BTC_Return, 1.0)

# Cumulative portfolio balance
backtest['Balance']         = initial_balance * backtest.Alg_Return.cumprod()
backtest['BuyHold_Balance'] = initial_balance * backtest.BTC_Return.cumprod()

backtest.tail()

In [ ]:
fig, ax = plt.subplots(dpi=150)

date_format = DateFormatter("%b-%d-%y")
ax.xaxis.set_major_formatter(date_format)
ax.tick_params(axis='x', labelsize=8)
fig.autofmt_xdate()

ax.plot(backtest['BuyHold_Balance'], lw=0.75, alpha=0.85, label='Buy and Hold')
ax.plot(backtest['Balance'],         lw=0.75, alpha=0.85, label='Crossing Averages Strategy')

ax.axhline(y=initial_balance, color='grey', linestyle='--', lw=0.6, label='Initial Capital')

ax.set_ylabel('Portfolio Value (USD)')
ax.set_title('Backtest: Crossing Averages vs Buy & Hold — 2020')
ax.grid(alpha=0.4)
ax.legend()

plt.tight_layout()
plt.savefig('charts/04_backtest_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Performance Summary

In [ ]:
final_algo      = backtest['Balance'].iloc[-1]
final_buyhold   = backtest['BuyHold_Balance'].iloc[-1]

algo_return     = (final_algo    - initial_balance) / initial_balance * 100
buyhold_return  = (final_buyhold - initial_balance) / initial_balance * 100

n_trades        = len(buy_signals) + len(sell_signals)

print("=" * 45)
print("         BACKTEST RESULTS — 2020")
print("=" * 45)
print(f"  Starting Capital   : ${initial_balance:>10,.2f}")
print(f"  MA Crossover Final : ${final_algo:>10,.2f}  ({algo_return:+.1f}%)")
print(f"  Buy & Hold Final   : ${final_buyhold:>10,.2f}  ({buyhold_return:+.1f}%)")
print(f"  Total Trades Made  : {n_trades}")
print(f"  SMA Periods Used   : {short_interval}-day / {long_interval}-day")
print("=" * 45)

---
## 9. Extension — Test on 2018–2019 (Bear Market)

Repeating the same strategy on a bear-market period to stress-test the algorithm.

In [ ]:
def run_backtest(ticker, start, end, short_window=10, long_window=40, capital=10_000.0, label=''):
    """Runs a Moving Average Crossover backtest and returns a summary dict."""
    data = yf.download(ticker, start=start, end=end, interval='1d')

    signals = pd.DataFrame(index=data.index)
    signals['Short']    = data['Close'].rolling(window=short_window, min_periods=1).mean()
    signals['Long']     = data['Close'].rolling(window=long_window,  min_periods=1).mean()
    signals['Signal']   = np.where(signals['Short'] > signals['Long'], 1.0, 0.0)
    signals['Position'] = signals['Signal'].diff()

    bt = pd.DataFrame(index=signals.index)
    bt['BTC_Return'] = data['Close'] / data['Close'].shift(1)
    bt['Alg_Return'] = np.where(signals.Signal == 1, bt.BTC_Return, 1.0)
    bt['Algo']       = capital * bt.Alg_Return.cumprod()
    bt['BuyHold']    = capital * bt.BTC_Return.cumprod()

    # Plot
    fig, axes = plt.subplots(2, 1, dpi=130, figsize=(9, 7))
    date_fmt  = DateFormatter("%b-%y")

    for ax in axes:
        ax.xaxis.set_major_formatter(date_fmt)
        ax.tick_params(axis='x', labelsize=7)
        fig.autofmt_xdate()

    axes[0].plot(data['Close'],    lw=0.7, label='Close')
    axes[0].plot(signals['Short'], lw=0.7, alpha=0.8, color='orange', label=f'{short_window}-Day SMA')
    axes[0].plot(signals['Long'],  lw=0.7, alpha=0.8, color='purple', label=f'{long_window}-Day SMA')
    axes[0].plot(signals[signals.Position ==  1.0].index,
                 signals.Short[signals.Position ==  1.0], marker=6, ms=5, ls='none', color='green', label='Buy')
    axes[0].plot(signals[signals.Position == -1.0].index,
                 signals.Short[signals.Position == -1.0], marker=7, ms=5, ls='none', color='red',   label='Sell')
    axes[0].set_title(f'{ticker} Price + Signals ({label})')
    axes[0].set_ylabel('USD')
    axes[0].legend(fontsize=7)
    axes[0].grid(alpha=0.4)

    axes[1].plot(bt['BuyHold'], lw=0.75, label='Buy & Hold')
    axes[1].plot(bt['Algo'],    lw=0.75, label='MA Crossover')
    axes[1].axhline(y=capital, color='grey', linestyle='--', lw=0.5)
    axes[1].set_title(f'Portfolio Value ({label})')
    axes[1].set_ylabel('USD')
    axes[1].legend(fontsize=7)
    axes[1].grid(alpha=0.4)

    plt.tight_layout()
    plt.savefig(f'charts/backtest_{label.replace(" ","_")}.png', dpi=130, bbox_inches='tight')
    plt.show()

    final_algo    = bt['Algo'].iloc[-1]
    final_buyhold = bt['BuyHold'].iloc[-1]
    return {
        'Period'         : label,
        'Algo_Return_%'  : round((final_algo    - capital) / capital * 100, 2),
        'BH_Return_%'    : round((final_buyhold - capital) / capital * 100, 2),
        'Algo_Final_$'   : round(final_algo,    2),
        'BH_Final_$'     : round(final_buyhold, 2),
    }

# Bear market test: 2018–2019
result_bear = run_backtest('BTC-USD', '2018-01-01', '2019-12-31', label='2018-2019')
print(result_bear)

In [ ]:
# Bull market test: 2020
result_bull = run_backtest('BTC-USD', '2020-01-01', '2020-12-31', label='2020')
print(result_bull)

In [ ]:
# Compare results
comparison = pd.DataFrame([result_bear, result_bull]).set_index('Period')
print("\nStrategy Comparison Across Periods")
print(comparison.to_string())

---
## 10. Key Takeaways

| Concept | Description |
|---|---|
| **Simple Moving Average** | Smooths price noise by averaging over a rolling window |
| **Crossover Signal** | Short SMA crossing above Long SMA → Buy; below → Sell |
| **Backtest** | Simulating strategy on historical data to assess viability |
| **Overfitting risk** | Tuning SMA parameters too tightly to historical data may not generalise |
| **Buy & Hold baseline** | Simple benchmark — just hold the asset for the entire period |

> ⚠️ **Disclaimer:** This project is for educational purposes only. Past performance does not guarantee future results. Do not use this as financial advice.

---
## References
- [yfinance Documentation](https://pypi.org/project/yfinance/)
- [Moving Average Crossover — Investopedia](https://www.investopedia.com/ask/answers/122314/how-do-i-use-moving-average-ma-create-forex-trading-strategy.asp)
- [Backtesting — Investopedia](https://www.investopedia.com/terms/b/backtesting.asp)
- [Moving Average Crossover — Wikipedia](https://en.wikipedia.org/wiki/Moving_average_crossover)
- IBM Skills Network Guided Project — David Pasternak (2021)